# Chapter 9.4 - Recurrent Neural Networks

A recurrent neural network keeps a hidden state that is updated at every time step. This notebook derives the recurrence, executes it by hand, turns hidden states into character logits, and makes the state-shape contract explicit.

## How to use this notebook

Run the notebook from top to bottom in a clean kernel. Everything is generated from small tensors or inline text, so there are no downloads. Before important cells, predict the time, batch, feature, vocabulary, and hidden-state shapes. Treat every assertion as an executable contract rather than decoration.

## You are done when you can

- contrast a next-token network with and without hidden state
- calculate a recurrent update one time step at a time
- identify input, hidden, and output parameter shapes
- explain how one set of parameters is reused across time
- diagnose a malformed initial hidden state


In [ ]:
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(0)
random.seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)


## 9.4.0 The Problem This Notebook Solves

A fixed-window model can use only the values explicitly placed in its input window. An RNN adds a **hidden state**: a learned numeric summary passed from one time step to the next.

For input `X_t` and previous state `H_(t-1)`, a simple RNN computes

```text
H_t = tanh(X_t W_xh + H_(t-1) W_hh + b_h)
O_t = H_t W_hq + b_q
```

`tanh` bounds each hidden value between -1 and 1. The logits `O_t` are unnormalized scores over possible next tokens. The same weight matrices are reused at every time step; recurrence does not create new parameters for every position.


## 9.4.1 Neural Networks Without Hidden States

A context-free network maps the current token to next-token logits. Two identical current tokens always produce identical logits, even if their earlier contexts differ. One-hot vectors make the lookup mechanism visible: a vector has length `vocab_size`, with one 1 at the token ID.


In [ ]:
vocab_size = 5
current_ids = torch.tensor([2, 2])
one_hot = F.one_hot(current_ids, num_classes=vocab_size).float()
W_xq = torch.arange(vocab_size * vocab_size, dtype=torch.float32).reshape(vocab_size, vocab_size)
context_free_logits = one_hot @ W_xq

print(one_hot)
print(context_free_logits)
assert shape(one_hot) == (2, vocab_size)
assert torch.equal(context_free_logits[0], context_free_logits[1])


## 9.4.2 Recurrent Networks With Hidden States

The batch below has shape `(batch=2, time=3)`. We transpose its one-hot representation to `(time, batch, vocab)` because the manual loop consumes one complete batch at each time step.

At each iteration, `X_t` changes and `H` is replaced by the new state. Parameters remain the same. The list `outputs` stores one logit tensor per time step.


In [ ]:
batch_ids = torch.tensor([[0, 1, 2], [3, 2, 1]])
inputs = F.one_hot(batch_ids.T, num_classes=vocab_size).float()
hidden_size = 4

W_xh = torch.randn(vocab_size, hidden_size) * 0.1
W_hh = torch.randn(hidden_size, hidden_size) * 0.1
b_h = torch.zeros(hidden_size)
W_hq = torch.randn(hidden_size, vocab_size) * 0.1
b_q = torch.zeros(vocab_size)
H = torch.zeros(batch_ids.shape[0], hidden_size)
outputs = []

for X_t in inputs:
    H = torch.tanh(X_t @ W_xh + H @ W_hh + b_h)
    outputs.append(H @ W_hq + b_q)

logits = torch.stack(outputs)
print("inputs:", shape(inputs), "state:", shape(H), "logits:", shape(logits))
assert shape(inputs) == (3, 2, 5)
assert shape(H) == (2, 4)
assert shape(logits) == (3, 2, 5)


## 9.4.3 Why Order Changes the State

The final hidden state is not a bag-of-tokens count. Swapping token order changes the sequence of nonlinear updates, even when the same tokens are present. This makes state useful for language, where `dog bites person` and `person bites dog` do not mean the same thing.


In [ ]:
def final_state(token_ids):
    state = torch.zeros(1, hidden_size)
    sequence = F.one_hot(torch.tensor(token_ids), num_classes=vocab_size).float()
    for X_t in sequence:
        state = torch.tanh(X_t.reshape(1, -1) @ W_xh + state @ W_hh + b_h)
    return state

state_forward = final_state([0, 1, 2])
state_reversed = final_state([2, 1, 0])
print(state_forward)
print(state_reversed)
assert not torch.allclose(state_forward, state_reversed)


## 9.4.4 RNN-Based Character-Level Language Models

A **character-level language model** receives a character ID at each time and predicts the following character. If logits have shape `(time, batch, vocab)`, flattening the first two axes gives `(time * batch, vocab)`. Labels must be flattened in the same time-major order.

Cross-entropy expects one logit row per target and a target integer ID per row.


In [ ]:
targets_batch_first = torch.tensor([[1, 2, 3], [2, 1, 0]])
targets_time_first = targets_batch_first.T
flat_logits = logits.reshape(-1, vocab_size)
flat_targets = targets_time_first.reshape(-1)
loss = F.cross_entropy(flat_logits, flat_targets)

print("flat logits:", shape(flat_logits), "flat targets:", shape(flat_targets))
print("loss:", loss.item())
assert shape(flat_logits) == (6, vocab_size)
assert shape(flat_targets) == (6,)
assert loss.ndim == 0


## 9.4.5 Break It Deliberately: Hidden-State Shape Mismatch

The hidden state needs one row per sequence in the batch and one column per hidden feature: `(batch, hidden_size)`. A state with three rows cannot be combined with a two-example input batch.


In [ ]:
X_t = inputs[0]
wrong_H = torch.zeros(3, hidden_size)
try:
    torch.tanh(X_t @ W_xh + wrong_H @ W_hh + b_h)
except RuntimeError as error:
    print(type(error).__name__)
    print(str(error).splitlines()[0])
else:
    raise AssertionError("Expected incompatible batch dimensions to fail")


## 9.4 Checkpoint

Answer these without rerunning the notebook. Short markdown answers are enough.

1. What information enters the hidden-state update at time t?
2. Which RNN quantities change each time step and which are reused?
3. Why can reversed token order produce a different final state?
4. What does each axis of `(time, batch, vocab)` mean?
5. Why must logits and labels use the same flattening order?
